# Daily Practice — 2026-10-17 — Statistics: Is Model B Actually Better, or Just Lucky?

**Dataset:** [UCI Wine Recognition dataset](https://archive.ics.uci.edu/dataset/109/wine) — 178 real
chemical analyses of wines grown in the same region of Italy, from three different cultivars (13 measured
features per wine: alcohol, malic acid, color intensity, proline, and so on). Loaded directly from
`sklearn.datasets.load_wine()`.

## Problem statement

You've moved from an SQA role onto an ML team. A teammate posts in Slack: "Prototyped a k-NN classifier as
a simpler drop-in for our Logistic Regression baseline on the wine-cultivar classifier — way less code,
no coefficients to maintain. Numbers looked fine on my laptop, let's swap it in." Your QA instinct kicks
in: "looked fine on my laptop" usually means "I ran it once." Before anyone merges the swap, you've been
asked to build the statistical tooling to answer, rigorously: *is the candidate model actually different
from the baseline, or is that one run just noise?*

**What you should produce:** four composable functions —

1. `get_cv_scores(model, X, y, cv)` — paired cross-validated accuracy scores for a model, using a `cv`
   splitter object built *once* and reused for every model, so every model sees exactly the same folds
   (that's what makes the comparison "paired").
2. `bootstrap_ci_diff(diffs, n_boot, alpha, random_state)` — a bootstrap confidence interval for the mean
   of a paired difference array.
3. `paired_significance_tests(diffs, scores_a, scores_b)` — a paired t-test and a Wilcoxon signed-rank
   test p-value for the same paired difference.
4. `mcnemar_test(y_true, pred_a, pred_b)` — McNemar's test on a single held-out test set, comparing the
   two models' *per-example* correctness (not their averaged scores).

— then apply all four to compare the Logistic Regression baseline against the k-NN candidate, and write a
short verdict: is the candidate significantly different from the baseline, and is there a reason the
different tests might not agree with each other?

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon, chi2
from sklearn.base import clone
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier  # noqa: F401 (handy if you want to extend the comparison)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 0

## 1. Load the data

TODO: implement `load_data()`. It should:
- call `sklearn.datasets.load_wine()`
- return `(X, y)` where `X` is a `pd.DataFrame` built from `data` with `feature_names` as columns, and
  `y` is a `pd.Series` built from `target`

No scaling, splitting, or cleaning here — this dataset is already numeric and complete. This function's
only job is to hand back a usable `(X, y)` pair.

In [ ]:
def load_data() -> tuple[pd.DataFrame, pd.Series]:
    """Load the UCI wine recognition dataset as (X, y)."""
    # TODO
    raise NotImplementedError

X, y = load_data()
X.shape, y.value_counts().sort_index()

## 2. Paired cross-validated scores

TODO: implement `get_cv_scores(model, X, y, cv)`. It should:
- accept a **fitted-from-scratch-each-fold** `model` (an unfitted estimator or `Pipeline`), `X`, `y`, and
  an already-constructed `cv` splitter object
- for each `(train_idx, test_idx)` yielded by `cv.split(X, y)`: clone the model with `sklearn.base.clone`
  (so each fold starts from an untrained copy), fit on the train indices, predict on the test indices, and
  record the fold's accuracy
- return a 1-D `np.ndarray` of per-fold accuracies, in the order `cv.split` produced them

Then:
- build `cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=RANDOM_STATE)` **once** —
  this is the crucial step that makes the later comparison "paired": both models must see the exact same
  folds
- build `lr_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))`
- build `knn_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`
- compute `lr_scores = get_cv_scores(lr_model, X, y, cv)` and `knn_scores = get_cv_scores(knn_model, X, y, cv)`
- compute `diffs = lr_scores - knn_scores`

Why clone the model and rebuild `cv` only once? If you built a *fresh* `cv` for each model (even with the
same `random_state` and parameters), you'd get the same fold indices — but calling `cv.split` twice on two
separately-constructed objects is fragile to rely on by accident. Building one `cv` and reusing it makes
the pairing explicit and impossible to get wrong.

In [ ]:
def get_cv_scores(model, X, y, cv) -> np.ndarray:
    """Per-fold accuracy of `model` over the folds in `cv`, using a fresh clone each fold."""
    # TODO
    raise NotImplementedError

# TODO: build cv, lr_model, knn_model, lr_scores, knn_scores, diffs
cv = None
lr_scores = None
knn_scores = None
diffs = None

## 3. Bootstrap confidence interval for the paired difference

TODO: implement `bootstrap_ci_diff(diffs, n_boot=10000, alpha=0.05, random_state=0)`. It should:
- use `np.random.default_rng(random_state)` to draw `n_boot` bootstrap resamples of `diffs` (same length
  as `diffs`, sampled with replacement)
- compute the mean of each resample
- return a 3-tuple `(observed_mean_diff, ci_lower, ci_upper)`, where the CI bounds are the
  `100*alpha/2` and `100*(1 - alpha/2)` percentiles of the bootstrap means

Run it on `diffs` from the previous section and look at whether the interval contains 0.

In [ ]:
def bootstrap_ci_diff(diffs: np.ndarray, n_boot: int = 10000, alpha: float = 0.05, random_state: int = 0):
    """Bootstrap CI for the mean of a paired-difference array."""
    # TODO
    raise NotImplementedError

# TODO: call bootstrap_ci_diff(diffs) and print the result

## 4. Paired hypothesis tests

TODO: implement `paired_significance_tests(lr_scores, knn_scores)`. It should:
- run `scipy.stats.ttest_rel(lr_scores, knn_scores)` (a paired t-test — assumes the per-fold differences
  are roughly normally distributed)
- run `scipy.stats.wilcoxon(lr_scores - knn_scores)` (a paired, non-parametric alternative with no
  normality assumption)
- return a `dict` with keys `ttest_stat`, `ttest_p`, `wilcoxon_stat`, `wilcoxon_p`

Call it and print the p-values next to the bootstrap CI from the previous section.

In [ ]:
def paired_significance_tests(scores_a: np.ndarray, scores_b: np.ndarray) -> dict:
    """Paired t-test and Wilcoxon signed-rank test for scores_a vs scores_b."""
    # TODO
    raise NotImplementedError

# TODO: call paired_significance_tests(lr_scores, knn_scores) and print the result

## 5. McNemar's test on a single held-out split

The sections above are all built on 100 *overlapping* cross-validation folds — great for squeezing a
precise-looking estimate out of 178 rows, but the folds share most of their data with each other, so they
are not independent observations. McNemar's test takes a different approach entirely: one honest
train/test split, and a test of whether the two models disagree about *which specific examples* they get
right.

TODO: implement `mcnemar_test(y_true, pred_a, pred_b)`. It should:
- build the "disagreement" counts: `n10` = examples model A got right and model B got wrong, `n01` =
  examples model A got wrong and model B got right
- if `n10 + n01 == 0` (the models never disagree), return `(0.0, 1.0, n10, n01)`
- otherwise compute the continuity-corrected McNemar statistic
  `stat = (abs(n10 - n01) - 1) ** 2 / (n10 + n01)` and its p-value `scipy.stats.chi2.sf(stat, df=1)`
- return `(stat, p_value, n10, n01)`

Then:
- split off a single held-out test set with `train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)`
- fit a fresh clone of `lr_model` and of `knn_model` on the training portion only
- predict both on the test portion
- call `mcnemar_test(y_test, lr_pred, knn_pred)`

In [ ]:
def mcnemar_test(y_true, pred_a, pred_b):
    """Continuity-corrected McNemar's test comparing per-example correctness of two models."""
    # TODO
    raise NotImplementedError

# TODO: train_test_split, fit lr_model/knn_model clones on train, predict on test, call mcnemar_test

## 6. Your verdict

TODO (written answer, 3-5 sentences): Based on the bootstrap CI, the two paired tests, and McNemar's
test — is the k-NN candidate significantly different from the Logistic Regression baseline? Do all four
results agree? If any of them disagree with the others, say which ones, and give one concrete reason why
(think about what's different between "100 CV folds" and "one 70/30 split" in terms of how much
independent information each one actually contains).

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon, chi2
from sklearn.base import clone
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 0


def load_data() -> tuple[pd.DataFrame, pd.Series]:
    """Load the UCI wine recognition dataset as (X, y)."""
    raw = load_wine()
    X = pd.DataFrame(raw.data, columns=raw.feature_names)
    y = pd.Series(raw.target, name="cultivar")
    return X, y


def get_cv_scores(model, X, y, cv) -> np.ndarray:
    """Per-fold accuracy of `model` over the folds in `cv`, using a fresh clone each fold."""
    X_arr, y_arr = np.asarray(X), np.asarray(y)
    scores = []
    for train_idx, test_idx in cv.split(X_arr, y_arr):
        fitted = clone(model).fit(X_arr[train_idx], y_arr[train_idx])
        preds = fitted.predict(X_arr[test_idx])
        scores.append((preds == y_arr[test_idx]).mean())
    return np.array(scores)


def bootstrap_ci_diff(diffs: np.ndarray, n_boot: int = 10000, alpha: float = 0.05, random_state: int = 0):
    """Bootstrap CI for the mean of a paired-difference array."""
    rng = np.random.default_rng(random_state)
    n = len(diffs)
    resample_idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = diffs[resample_idx].mean(axis=1)
    lo, hi = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return diffs.mean(), lo, hi


def paired_significance_tests(scores_a: np.ndarray, scores_b: np.ndarray) -> dict:
    """Paired t-test and Wilcoxon signed-rank test for scores_a vs scores_b."""
    t_stat, t_p = ttest_rel(scores_a, scores_b)
    w_stat, w_p = wilcoxon(scores_a - scores_b)
    return {"ttest_stat": t_stat, "ttest_p": t_p, "wilcoxon_stat": w_stat, "wilcoxon_p": w_p}


def mcnemar_test(y_true, pred_a, pred_b):
    """Continuity-corrected McNemar's test comparing per-example correctness of two models."""
    y_true, pred_a, pred_b = np.asarray(y_true), np.asarray(pred_a), np.asarray(pred_b)
    correct_a, correct_b = pred_a == y_true, pred_b == y_true
    n10 = int(np.sum(correct_a & ~correct_b))
    n01 = int(np.sum(~correct_a & correct_b))
    if n10 + n01 == 0:
        return 0.0, 1.0, n10, n01
    stat = (abs(n10 - n01) - 1) ** 2 / (n10 + n01)
    return stat, chi2.sf(stat, df=1), n10, n01


# --- Run the comparison: Logistic Regression baseline vs. k-NN candidate ---
X, y = load_data()

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=RANDOM_STATE)
lr_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
knn_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

lr_scores = get_cv_scores(lr_model, X, y, cv)
knn_scores = get_cv_scores(knn_model, X, y, cv)
diffs = lr_scores - knn_scores

print(f"LR mean CV accuracy:  {lr_scores.mean():.4f}")
print(f"kNN mean CV accuracy: {knn_scores.mean():.4f}")

mean_diff, ci_lo, ci_hi = bootstrap_ci_diff(diffs)
print(f"Bootstrap 95% CI for (LR - kNN): {mean_diff:.4f}  [{ci_lo:.4f}, {ci_hi:.4f}]")

tests = paired_significance_tests(lr_scores, knn_scores)
print(f"Paired t-test:        p = {tests['ttest_p']:.5f}")
print(f"Wilcoxon signed-rank: p = {tests['wilcoxon_p']:.5f}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)
lr_fit = clone(lr_model).fit(X_train, y_train)
knn_fit = clone(knn_model).fit(X_train, y_train)
lr_pred, knn_pred = lr_fit.predict(X_test), knn_fit.predict(X_test)

mc_stat, mc_p, n10, n01 = mcnemar_test(y_test, lr_pred, knn_pred)
print(f"McNemar (single 70/30 split): stat = {mc_stat:.3f}, p = {mc_p:.4f}  (n10={n10}, n01={n01})")
```

**Typical output** (`RANDOM_STATE = 0`):

```text
LR mean CV accuracy:  0.9810
kNN mean CV accuracy: 0.9651
Bootstrap 95% CI for (LR - kNN): 0.0159  [0.0079, 0.0242]
Paired t-test:        p = 0.00023
Wilcoxon signed-rank: p = 0.00013
McNemar (single 70/30 split): stat = 0.500, p = 0.4795  (n10=2, n01=0)
```

**The verdict, and why the tests don't agree.** The repeated-CV-based bootstrap CI and both paired tests
all say the same confident thing: LR beats kNN by about 1.6 points, and that gap is very unlikely to be
noise (`p < 0.001`, and the CI for the difference excludes 0). McNemar's test, run on one honest 70/30
split, says the opposite: with only 54 held-out examples and a difference of just two misclassified
rows, `p = 0.48` — nowhere near significant.

Both answers are "correct" given what each test actually measures, and that's the catch an SQA background
should be primed to look for:

- **The 100 "paired" CV folds are not 100 independent experiments.** Repeated k-fold CV reuses the same
  178 rows over and over across folds and repeats, so the fold-to-fold scores are correlated. Treating
  them as i.i.d. observations in a paired t-test (or Wilcoxon test) understates the true variance and
  produces p-values that are *anti-conservative* — smaller, and more "significant-looking," than they
  should be. This is a well-known pitfall in the ML evaluation literature (Dietterich, 1998); a corrected
  variance estimator (e.g. Nadeau & Bengio's corrected resampled t-test) exists specifically to fix it,
  and a production MLQA pipeline comparing models should use one rather than a plain `ttest_rel` on raw CV
  folds.
- **McNemar's test on one split is valid but severely underpowered here.** It only uses the 54 held-out
  examples from a single split, and with accuracy already near-ceiling (`LR = 1.00`, `kNN = 0.96` on that
  split), there are only `2` disagreeing examples total — nowhere near enough to detect a 1.6-point
  population-level gap reliably. A non-significant McNemar result here means "this one split didn't have
  enough disagreeing examples to tell," not "the models are the same."
- **Practical takeaway:** don't report a single p-value as if it settles the question. The honest
  statement is "the CV-based evidence points to a real, reproducible gap favoring LR, but it's likely
  overconfident because of fold non-independence; the single-split McNemar test is underpowered and can't
  confirm or refute that on its own — we'd want either a corrected-variance test or a second, truly
  independent held-out split before blocking the kNN swap on these numbers alone." Flagging *that*
  tension — rather than picking whichever test gives the answer you expected — is the actual QA skill
  being exercised here.

</details>